In [3]:
import re
import numpy as np
import pandas as pd

from scipy.interpolate import PchipInterpolator

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [4]:
df = pd.read_csv("dataset.csv")

print(df.shape)

df.head()

(975, 30)


,datetime,underlying_price,NIFTY27JAN2625200CE,NIFTY27JAN2625300CE,NIFTY27JAN2625400CE,NIFTY27JAN2625500CE,NIFTY27JAN2625600CE,NIFTY27JAN2625700CE,NIFTY27JAN2625800CE,NIFTY27JAN2625900CE,...,NIFTY27JAN2624200PE,NIFTY27JAN2624300PE,NIFTY27JAN2624400PE,NIFTY27JAN2624500PE,NIFTY27JAN2624600PE,NIFTY27JAN2624700PE,NIFTY27JAN2624800PE,NIFTY27JAN2624900PE,NIFTY27JAN2625000PE,NIFTY27JAN2625100PE
0,07-01-2026 09:15,26111.65,0.12662,0.12330,0.11741,NaN,0.11005,0.10576,NaN,0.09724,...,0.15760,0.15240,0.14697,0.14105,0.13613,0.13085,0.12640,0.12142,0.11631,0.11150
1,07-01-2026 09:20,26141.40,0.08632,NaN,NaN,0.11779,0.11197,0.11028,NaN,NaN,...,NaN,0.15420,0.14753,0.14274,0.13849,0.13282,NaN,0.12363,NaN,0.11353
2,07-01-2026 09:25,26139.35,0.09147,NaN,0.09514,0.09933,0.09599,0.09204,0.09216,0.08954,...,0.15927,NaN,0.14919,0.14245,0.13806,0.13242,0.12877,0.12349,0.11817,NaN
3,07-01-2026 09:30,26128.95,0.10860,0.10842,0.11150,0.12248,0.10715,0.11098,0.10345,NaN,...,0.15755,NaN,0.14691,0.14209,0.13721,0.13184,0.12722,0.12252,0.11729,0.11200
4,07-01-2026 09:35,26131.90,0.10462,0.10538,0.12459,0.12051,0.11225,0.11294,0.10544,NaN,...,0.15924,0.15334,0.14784,0.14230,NaN,0.13219,0.12733,0.12295,0.11707,NaN


In [5]:
option_cols = [
    c for c in df.columns
    if c not in ["datetime", "underlying_price"]
]

print("Option Columns:", len(option_cols))

Option Columns: 28


In [6]:
def get_strike(col):

    nums = re.findall(r"\d+", col)

    return int(nums[-1])


strike_map = {
    c: get_strike(c)
    for c in option_cols
}

In [7]:
call_cols = [
    c for c in option_cols
    if c.endswith("CE")
]

put_cols = [
    c for c in option_cols
    if c.endswith("PE")
]

call_cols = sorted(
    call_cols,
    key=lambda x: strike_map[x]
)

put_cols = sorted(
    put_cols,
    key=lambda x: strike_map[x]
)

print(len(call_cols))
print(len(put_cols))

14
14


In [8]:
filled_df = df.copy()


def fill_surface_row(
    row,
    columns
):

    strikes = np.array(
        [strike_map[c] for c in columns]
    )

    values = row[columns].values.astype(float)

    mask = ~np.isnan(values)

    if mask.sum() < 3:
        return values

    try:

        pchip = PchipInterpolator(
            strikes[mask],
            values[mask],
            extrapolate=True
        )

        values[~mask] = pchip(
            strikes[~mask]
        )

    except:
        pass

    return values

In [9]:
for idx in filled_df.index:

    filled_df.loc[idx, call_cols] = (
        fill_surface_row(
            filled_df.loc[idx],
            call_cols
        )
    )

    filled_df.loc[idx, put_cols] = (
        fill_surface_row(
            filled_df.loc[idx],
            put_cols
        )
    )

print("PCHIP completed")

PCHIP completed


In [10]:
for col in option_cols:

    filled_df[col] = (
        filled_df[col]
        .interpolate(
            method="linear",
            limit_direction="both"
        )
    )

print("Time interpolation completed")

Time interpolation completed


In [11]:
surface = filled_df.drop(
    columns=["datetime"]
)

imp = IterativeImputer(
    max_iter=20,
    random_state=42
)

filled_surface = imp.fit_transform(
    surface
)

filled_surface = pd.DataFrame(
    filled_surface,
    columns=surface.columns
)

print("Iterative Imputer completed")

Iterative Imputer completed


In [12]:
final_df = filled_df.copy()

final_df.loc[
    :,
    surface.columns
] = filled_surface

final_df.head()

,datetime,underlying_price,NIFTY27JAN2625200CE,NIFTY27JAN2625300CE,NIFTY27JAN2625400CE,NIFTY27JAN2625500CE,NIFTY27JAN2625600CE,NIFTY27JAN2625700CE,NIFTY27JAN2625800CE,NIFTY27JAN2625900CE,...,NIFTY27JAN2624200PE,NIFTY27JAN2624300PE,NIFTY27JAN2624400PE,NIFTY27JAN2624500PE,NIFTY27JAN2624600PE,NIFTY27JAN2624700PE,NIFTY27JAN2624800PE,NIFTY27JAN2624900PE,NIFTY27JAN2625000PE,NIFTY27JAN2625100PE
0,07-01-2026 09:15,26111.65,0.12662,0.123300,0.117410,0.113567,0.11005,0.10576,0.100968,0.097240,...,0.157600,0.152400,0.14697,0.14105,0.136130,0.13085,0.126400,0.12142,0.116310,0.111500
1,07-01-2026 09:20,26141.40,0.08632,0.104578,0.114681,0.117790,0.11197,0.11028,0.106801,0.102157,...,0.160026,0.154200,0.14753,0.14274,0.138490,0.13282,0.128144,0.12363,0.118696,0.113530
2,07-01-2026 09:25,26139.35,0.09147,0.092704,0.095140,0.099330,0.09599,0.09204,0.092160,0.089540,...,0.159270,0.154267,0.14919,0.14245,0.138060,0.13242,0.128770,0.12349,0.118170,0.112810
3,07-01-2026 09:30,26128.95,0.10860,0.108420,0.111500,0.122480,0.10715,0.11098,0.103450,0.100809,...,0.157550,0.151904,0.14691,0.14209,0.137210,0.13184,0.127220,0.12252,0.117290,0.112000
4,07-01-2026 09:35,26131.90,0.10462,0.105380,0.124590,0.120510,0.11225,0.11294,0.105440,0.101709,...,0.159240,0.153340,0.14784,0.14230,0.137153,0.13219,0.127330,0.12295,0.117070,0.109909


In [13]:
final_df.to_csv(
    "filled_dataset_final_attempt.csv",
    index=False
)

print("Saved")

Saved


In [14]:
import pandas as pd

ORIGINAL_DATASET_PATH = "dataset.csv"

SEPARATOR = "||"

def generate_solution(
    filled_path,
    output_path
):

    original = pd.read_csv(
        ORIGINAL_DATASET_PATH
    )

    filled = pd.read_csv(
        filled_path
    )

    feature_cols = [
        c for c in original.columns
        if c != "datetime"
    ]

    rows = []

    for col in feature_cols:

        was_missing = (
            original[col].isna()
        )

        for idx in original.index[
            was_missing
        ]:

            dt = original.loc[
                idx,
                "datetime"
            ]

            uid = (
                f"{dt}{SEPARATOR}{col}"
            )

            val = filled.loc[
                idx,
                col
            ]

            rows.append(
                {
                    "id": uid,
                    "value": val
                }
            )

    solution = pd.DataFrame(
        rows
    )

    solution = (
        solution
        .sort_values("id")
        .reset_index(drop=True)
    )

    solution.to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved {output_path}"
    )

In [15]:
generate_solution(
    "filled_dataset_final_attempt.csv",
    "submission_final_attempt.csv"
)

Saved submission_final_attempt.csv


In [16]:
sub = pd.read_csv(
    "submission_final_attempt.csv"
)

print(sub.shape)

sub.head()

(5460, 2)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2624100PE,0.163830
1,07-01-2026 09:15||NIFTY27JAN2625500CE,0.113567
2,07-01-2026 09:15||NIFTY27JAN2625800CE,0.100968
3,07-01-2026 09:20||NIFTY27JAN2624000PE,0.170134
4,07-01-2026 09:20||NIFTY27JAN2624200PE,0.160026
